In [1]:
# ===== Cell 1: 爬取知乎文章内容 =====
# 知乎专栏文章 URL
# 用 requests + BeautifulSoup 抓取，提取 class="Post-RichText" 下的正文
# 注意：知乎有反爬机制，如果 requests 被 403 拦截，自动回退到本地缓存文件
import os
import requests
from bs4 import BeautifulSoup

url = 'https://zhuanlan.zhihu.com/p/333623112'

# 知乎需要浏览器 User-Agent 才不会直接 403
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
    'Referer': 'https://www.zhihu.com/',
}

article_text = ''

try:
    resp = requests.get(url, headers=headers, timeout=30)
    resp.encoding = 'utf-8'
    print(f'HTTP 状态码: {resp.status_code}')

    if resp.status_code == 200:
        soup = BeautifulSoup(resp.text, 'lxml')
        article_el = soup.find(class_='Post-RichText')
        if not article_el:
            article_el = soup.find(class_='RichText')
        if article_el:
            raw_text = article_el.get_text(separator='\n', strip=True)
            lines = [ln.strip() for ln in raw_text.split('\n') if ln.strip()]
            article_text = '\n'.join(lines)
            print(f'爬取成功: {len(article_text)} 字符, {len(lines)} 段')
        else:
            print('页面中未找到 Post-RichText 元素')
    else:
        print(f'知乎反爬拦截，状态码 {resp.status_code}')
except Exception as e:
    print(f'爬取失败: {e}')

# 回退：从本地缓存文件加载（如果爬取失败或内容为空）
if not article_text:
    cache_path = os.path.join(os.getcwd(), 'article_333623112.txt')
    if os.path.exists(cache_path):
        with open(cache_path, 'r', encoding='utf-8') as f:
            article_text = f.read()
        print(f'已从本地缓存加载: {len(article_text)} 字符')
    else:
        print('本地缓存文件也不存在，请检查！')

print(f'\n最终文本: {len(article_text)} 字符')
print(f'前200字预览:\n{article_text[:200]}')


HTTP 状态码: 403
知乎反爬拦截，状态码 403
已从本地缓存加载: 5536 字符

最终文本: 5536 字符
前200字预览:
50个科学小知识
1、兔子用腿拍打地面的动作来传递信息,用后腿拍打地面的大多数是雄兔,这是它向雌兔表达情感的一种方式.
2、世界上最大的猴是狒狒,最小的猴子是倭狨.
3、"四不象"真正的名字叫麋鹿,是我国的珍奇动物.
4、冰糕为什么会冒气？
冰糕冒气是因为外界空气中有不少眼睛看不见的水汽，碰到很冷的冰糕时，一遇冷就液化成雾滴包围在冰糕周围，看上去似乎是冰糕在"冒气"一样。
5、向日葵为什么总是向着


In [2]:
# ===== Cell 2: 加载本地 bge-large-zh 嵌入模型 =====
import os
import subprocess
import time
import torch
from transformers import AutoTokenizer, AutoModel
from langchain_core.embeddings import Embeddings

# 自动创建 X: 盘（解决中文路径问题）
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists('X:\\'):
    subprocess.run(['subst', 'X:', project_root], check=False)

EMBED_PATH = r'X:\6.transformers\models\bge-large-zh-v1.5'


class BGEEmbeddings(Embeddings):
    '''本地 bge-large-zh 向量模型: CLS 池化 + L2 归一化, 输出 1024 维'''

    def __init__(self, model_path, device='cpu'):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModel.from_pretrained(model_path).to(device).eval()

    def _embed(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        with torch.no_grad():
            inputs = self.tokenizer(
                texts, padding=True, truncation=True,
                max_length=512, return_tensors='pt',
            ).to(self.device)
            outputs = self.model(**inputs)
            cls = outputs.last_hidden_state[:, 0]
            cls = torch.nn.functional.normalize(cls, p=2, dim=1)
        return cls.cpu().tolist()

    def embed_documents(self, texts):
        return self._embed(texts)

    def embed_query(self, text):
        return self._embed(text)[0]


embeddings = BGEEmbeddings(EMBED_PATH, device='cpu')

t0 = time.time()
vec = embeddings.embed_query('测试向量')
print(f'嵌入模型就绪: 维度={len(vec)}, 耗时={time.time()-t0:.2f}s')



S:\PythonAILearningCourse\15-transformers-basics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11990.47it/s]


嵌入模型就绪: 维度=1024, 耗时=0.26s


In [3]:
# ===== Cell 3: 文本分块 + 构建 FAISS 向量库 =====
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1) 将爬取的整篇文章按段落拆成 Document 列表
# 每段作为一个独立的知识条目
paragraphs = [ln for ln in article_text.split('\n') if len(ln) > 10]
docs = [Document(page_content=p, metadata={'source': url, 'para': i})
        for i, p in enumerate(paragraphs)]
print(f'原始段落数: {len(docs)}')

# 2) 长段落进一步切分（bge 最大 512 token, 约 300~400 中文字符）
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n', '。', '；', '，', ' '],
)
chunks = splitter.split_documents(docs)
print(f'分块后文档数: {len(chunks)}')

# 3) 构建 FAISS 向量库
t0 = time.time()
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f'FAISS 向量库构建完成: {len(chunks)} 个块, 耗时 {time.time()-t0:.1f}s')



C:\Users\rehuanoo\AppData\Local\Temp\ipykernel_35732\2780014969.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


原始段落数: 77
分块后文档数: 77
FAISS 向量库构建完成: 77 个块, 耗时 21.2s


In [4]:
# ===== Cell 4: 相似度检索测试 =====
# 测试几个问题，看能否从知乎文章中检索到相关内容
questions = [
    '为什么向日葵总是向着太阳？',
    '为什么星星会一闪一闪的？',
    '为什么人老了头发会变白？',
    '为什么海水是蓝色的？',
]

for q in questions:
    results = vectorstore.similarity_search_with_score(q, k=2)
    print(f'\n问: {q}')
    for i, (doc, score) in enumerate(results, 1):
        print(f'  [{i}] score={score:.4f} | {doc.page_content[:80]}...')




问: 为什么向日葵总是向着太阳？
  [1] score=0.3386 | 5、向日葵为什么总是向着太阳?...
  [2] score=0.8085 | 向日葵的茎部含有一种奇妙的植物生长素。这种生长素非常怕光。一遇光线照射，它就会到背光的一面去，同时它还刺激背光一面的细胞迅速繁殖，所以，背光的一面就比向光的一面...

问: 为什么星星会一闪一闪的？
  [1] score=0.3049 | 8、为什么星星会一闪一闪的？...
  [2] score=0.8416 | 我们看到星闪闪，这不是因为星星本身的光度出现变化，而是与大气的遮挡有关。大气隔在我们与星星之间，当星光通过大气层时，会受到大气的密度和厚薄影响。大气不是绝对的透...

问: 为什么人老了头发会变白？
  [1] score=0.4196 | 11、为什么人老了头发便会变白？...
  [2] score=0.6079 | 我们的头发中有一种叫「黑色素」的物质，黑色素愈多头发的颜色便愈黑。而黑色素少的话，头发便会发黄或变白。人类到了老年时，身体的各种机能会逐渐衰退，色素的形成亦会愈...

问: 为什么海水是蓝色的？
  [1] score=0.7003 | 21、为什么海水大多是蓝、绿色？...
  [2] score=0.8464 | 望向大海，很多时也发现海水呈现蓝、绿色。可是，当你把海水捞起时，你却只能看到它像往日的水般，透明无色。原来，海水本身与我们日常所接触到的水没有大分别，也是透明的...


In [5]:
def ask(problem):
    results = vectorstore.similarity_search_with_score(problem, k=2)
    print(f'\n问: {problem}')
    for i, (doc, score) in enumerate(results, 1):
        print(f'  [{i}] score={score:.4f} | {doc.page_content[:80]}...')
    

In [6]:
ask('婴儿为什么呼吸？')


问: 婴儿为什么呼吸？
  [1] score=0.7797 | 婴儿刚出生时都会呱呱大哭，这不是因为他们感到不开心，而是他们正在大口大口地呼吸着第一口的空气呢！当婴儿离开妈妈身体出生时，他们吸进的第一口空气会冲到喉部去，这会...
  [2] score=0.8547 | 16、为什么婴儿刚出生时都会哭个不停？...
